### Merge all prediction results

The jupyter notebook contains code to merge all of the model results (DNN, XGB, additive PGS, dominance-adjusted PGS) into tables so they can be plotted. The two tables created here can be loaded into the ```07b_figure2.R``` and ```07c_figure3.R``` files to recreate the figures 2a, 2b, and 3b. 

To remake this table, download the results from GitHub (`/results/model_out/`) and set the ```base_path``` in cell #2 to your own filepath. 

#### DATA & CODE
GitHub + Readme: https://github.com/nybell/non-add-paper/tree/main         
Zenodo repo: https://zenodo.org/records/17552313                                                 
Manuscript DOI: https://doi.org/10.1101/2025.10.10.25337750                            
Questions: n.y.bell@vu.nl                       

______________________________________________________________________________

In [1]:
# import packages
import re
import os
import pandas as pd


In [ ]:
# Base directories
base_path = "/path/to/model_out/"                           # EDIT TO YOUR OWN FILE PATH
prs_dir = os.path.join(base_path, "regression_out")
xgb_dir = os.path.join(base_path, "xgb_out")
dnn_dir = os.path.join(base_path, "dnn_out")
devsim_dir = os.path.join(base_path, "devsim")

In [3]:
# Step 1: Define file loading functions
def load_r2_from_txt(file_path):
    """Load R² values from a .txt file."""
    df = pd.read_csv(file_path, header='infer', sep='\t')  # Load the DataFrame
    return df['r2'].tolist()

def load_r2_from_pkl(file_path):
    """Load R² values from a DataFrame in a .pkl file."""
    df = pd.read_pickle(file_path)  # Load the DataFrame
    if 'r2' in df.columns:
        return df['r2'].tolist()
    else:
        raise KeyError(f"'r2' column not found in {file_path}")

def extract_phenotype(file_name):
    """Extract phenotype from the file name."""
    parts = file_name.split("_")
    # Find the starting part with "h" and combine up to the "d" part
    for i, part in enumerate(parts):
        if part.startswith("h") and i + 2 < len(parts):
            return "_".join(parts[i:i+3])  # Example: h0.5_a0_d0.5
    return "unknown"

def extract_nsnps(file_name):
    """
    Extract the number following 'nsnps' in the file name.
    For example, 'DNN_nsnps100_h0.1_a0_d0.1_50k.pkl' returns 100.
    """
    match = re.search(r'nsnps(\d+)', file_name)
    if match:
        return int(match.group(1))
    else:
        return None
    
def extract_devsim(file_name):
    """
    Extract the number following 'devsim_' in the file name.
    For example, 'DNN_nsnps100_h0.1_a0_d0.1_50k.pkl' returns 100.
    """
    match = re.search(r'devsim_(\d+(?:\.\d+)?)', file_name)
    if match:
        return float(match.group(1))
    else:
        return None

def extract_prop_add(phenotype):
    """Extract prop_add (number after 'a') from phenotype."""
    try:
        return float(phenotype.split("_")[1][1:])  # Extract number after 'a'
    except IndexError:
        return None

def extract_prop_dom(phenotype):
    """Extract prop_dom (number after 'd') from phenotype."""
    try:
        return float(phenotype.split("_")[2][1:])  # Extract number after 'd'
    except IndexError:
        return None

In [4]:
model_results = []  # To store results

# Load R² values from .txt files in prs_out
for file in os.listdir(prs_dir):
    if file.startswith("ADD_R2") or file.startswith("DOM_R2"):
        file_path = os.path.join(prs_dir, file)
        model_type = "ADD" if file.startswith("ADD_R2") else "DOM"
        phenotype = extract_phenotype(file)
        nsnps = extract_nsnps(file)
        r2_values = load_r2_from_txt(file_path)
        model_results.append({
            'Model': model_type,
            'Phenotype': phenotype,
            'Nsnps': nsnps,
            'R2': r2_values
        })

# Load R² values from .pkl files in xgb_out and dnn_out
for model_dir, model_name in [(xgb_dir, 'XGB'), (dnn_dir, 'DNN')]:
    for file in os.listdir(model_dir):
        if file.endswith('.pkl'):
            file_path = os.path.join(model_dir, file)
            phenotype = extract_phenotype(file)
            nsnps = extract_nsnps(file)
            r2_values = load_r2_from_pkl(file_path)
            model_results.append({
                'Model': model_name,
                'Phenotype': phenotype,
                'Nsnps': nsnps,
                'R2': r2_values
            })

# Step 3: Convert results to DataFrame
data = []
for result in model_results:
    for r2 in result['R2']:
        data.append({'Model': result['Model'], 'Phenotype': result['Phenotype'], 'Nsnps': result['Nsnps'], 'R2': r2})

df = pd.DataFrame(data)

# Ensure prop_add and prop_dom are extracted and numeric
df['prop_add'] = pd.to_numeric(df['Phenotype'].apply(extract_prop_add), errors="coerce")
df['prop_dom'] = pd.to_numeric(df['Phenotype'].apply(extract_prop_dom), errors="coerce")

# Create a mapping for pheno_name based on prop_add and prop_dom
phenotype_order = df.groupby('Phenotype')['prop_add'].mean().sort_values(ascending=False).index
pheno_mapping = {
    phenotype: f"ADD_{extract_prop_add(phenotype)}_DEV_{extract_prop_dom(phenotype)}"
    for phenotype in phenotype_order
}

# Map the Phenotype column to pheno_name
df['pheno_name'] = df['Phenotype'].map(pheno_mapping)

# create a new column for total proportion
df['h2_total'] = df['prop_add'] + df['prop_dom']

# make column with ratio of dominance to total h2
df['pheno_type'] = (df['prop_dom'] / df['h2_total']).round(3)

# check data
df.head()

,Model,Phenotype,Nsnps,R2,prop_add,prop_dom,pheno_name,h2_total,pheno_type
0,DOM,h0.5_a0.25_d0.25,500,0.486187,0.25,0.25,ADD_0.25_DEV_0.25,0.5,0.5
1,DOM,h0.5_a0.25_d0.25,500,0.497102,0.25,0.25,ADD_0.25_DEV_0.25,0.5,0.5
2,DOM,h0.5_a0.25_d0.25,500,0.477112,0.25,0.25,ADD_0.25_DEV_0.25,0.5,0.5
3,DOM,h0.5_a0.25_d0.25,500,0.489030,0.25,0.25,ADD_0.25_DEV_0.25,0.5,0.5
4,DOM,h0.5_a0.25_d0.25,500,0.481713,0.25,0.25,ADD_0.25_DEV_0.25,0.5,0.5


In [ ]:
# Save DataFrame to Excel
df.to_excel("/path/to/fig2_data_aug2025.xlsx", index=False)  # SET OWN FILE PATH

#### DEVSIM results
 
This code block store the results for the different size dominance deviation simulations. 

In [6]:
devsim_results = []  # To store results

# Load R² values from .txt files in prs_out
for file in os.listdir(devsim_dir):
    if file.startswith("ADD_R2") or file.startswith("DOM_R2"):
        file_path = os.path.join(devsim_dir, file)
        model_type = "ADD" if file.startswith("ADD_R2") else "DOM"
        phenotype = extract_phenotype(file)
        nsnps = extract_nsnps(file)
        devsim = extract_devsim(file)
        r2_values = load_r2_from_txt(file_path)
        devsim_results.append({
            'Model': model_type,
            'Phenotype': phenotype,
            'Nsnps': nsnps,
            'devsim': devsim,
            'R2': r2_values
        })

# 
for model_name in ['XGB', 'DNN']:
    for file in os.listdir(devsim_dir):
        if file.endswith('.pkl') and file.startswith(model_name):
            file_path = os.path.join(devsim_dir, file)
            phenotype = extract_phenotype(file)
            nsnps = extract_nsnps(file)
            devsim = extract_devsim(file)
            r2_values = load_r2_from_pkl(file_path)
            devsim_results.append({
                'Model': model_name,
                'Phenotype': phenotype,
                'Nsnps': nsnps,
                'devsim': devsim,
                'R2': r2_values
            })

# Step 3: Convert results to DataFrame
devsim_data = []
for result in devsim_results:
    for r2 in result['R2']:
        devsim_data.append({'Model': result['Model'], 'Phenotype': result['Phenotype'],
                             'Nsnps': result['Nsnps'], 'devsim': result['devsim'], 'R2': r2})

dev_df = pd.DataFrame(devsim_data)

# Ensure prop_add and prop_dom are extracted and numeric
dev_df['prop_add'] = pd.to_numeric(dev_df['Phenotype'].apply(extract_prop_add), errors="coerce")
dev_df['prop_dom'] = pd.to_numeric(dev_df['Phenotype'].apply(extract_prop_dom), errors="coerce")

# Create a mapping for pheno_name based on prop_add and prop_dom
phenotype_order = dev_df.groupby('Phenotype')['prop_add'].mean().sort_values(ascending=False).index
pheno_mapping = {
    phenotype: f"ADD_{extract_prop_add(phenotype)}_DEV_{extract_prop_dom(phenotype)}"
    for phenotype in phenotype_order
}

# Map the Phenotype column to pheno_name
dev_df['pheno_name'] = dev_df['Phenotype'].map(pheno_mapping)

# create a new column for total proportion
dev_df['h2_total'] = dev_df['prop_add'] + dev_df['prop_dom']

# make column with ratio of dominance to total h2
dev_df['pheno_type'] = (dev_df['prop_dom'] / dev_df['h2_total']).round(3)

# check data
dev_df.head()

,Model,Phenotype,Nsnps,devsim,R2,prop_add,prop_dom,pheno_name,h2_total,pheno_type
0,DOM,h0.2_a0.15_d0.05,100,0.2,0.204769,0.15,0.05,ADD_0.15_DEV_0.05,0.2,0.25
1,DOM,h0.2_a0.15_d0.05,100,0.2,0.197265,0.15,0.05,ADD_0.15_DEV_0.05,0.2,0.25
2,DOM,h0.2_a0.15_d0.05,100,0.2,0.200385,0.15,0.05,ADD_0.15_DEV_0.05,0.2,0.25
3,DOM,h0.2_a0.15_d0.05,100,0.2,0.184414,0.15,0.05,ADD_0.15_DEV_0.05,0.2,0.25
4,DOM,h0.2_a0.15_d0.05,100,0.2,0.198175,0.15,0.05,ADD_0.15_DEV_0.05,0.2,0.25


In [15]:
# compute dominance deviation ratio k
# k = d / a
# d = Aa - ((AA = aa) / 2)

# define d
dev_df['d'] = (dev_df['devsim'] - 0.5).round(3)

# define k
dev_df['k'] = (dev_df['d'] / 0.5).round(3)

# check 
print(dev_df['k'].sort_values(ascending=False).unique())

[ 0.  -0.2 -0.4 -0.6 -0.8 -1. ]


In [ ]:
# Save DataFrame to Excel
dev_df.to_excel("/path/to/fig3b_data_aug2025.xlsx", index=False)             # SET OWN FILE PATH